# Somo 09 - Mfano wa Ubunifu wa Metakognisheni


## Usanidi

Daftari hili linaonyesha mfano wa kubuni wa Metacognition ukitumia Microsoft Agent Framework.

**Mahitaji:**
- Utekelezaji wa Azure OpenAI umewekwa kupitia vigezo vya mazingira
- Azure CLI imeidhinishwa (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Metacognition ni nini?

Metacognition ni **kufikiri kuhusu kufikiri**. Katika muktadha wa mawakala wa AI, ina maana ya kujenga mawakala ambao wanaweza:

- **Kujitafakari** juu ya matokeo yao wenyewe na mchakato wao wa kufikiria
- **Kutambua makosa** na kupona kwa heshima badala ya kushindwa kimya
- **Kutathmini** ikiwa majibu yao ni kamili na ya msaada
- **Kubadilisha** mkakati wao wakati mbinu ya awali haifanyi kazi (kwa mfano, kurudi kwa mfumo mbadala)

Wakala wa metacognition si tu hujibu maswali — hufuatilia utendaji wake mwenyewe na kurekebisha papo hapo.


## Zana za Msingi na za Akiba

Mfano wa kawaida wa metakognisheni ni **mkakati wa kugeuka kwa zana mbadala**. Wakala hujaribu kwanza zana ya msingi; ikiwa itaanguka (kwa mfano, hitilafu ya 404), wakala atatambua kushindwa na kwa uwazi awabadilishe kwa zana ya akiba.

Hii inaakisi mifumo ya ulimwengu halisi ambapo huduma za msingi zinaweza kutokuwepo na mawakala lazima wajitathmini tatizo wenyewe kabla ya kuchagua njia mbadala.

Below we define two flight lookup tools:
- **Msingi** — inafunika Paris, Tokyo, na Barcelona
- **Akiba** — inafunika Berlin, Sydney, na New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Wakala Mwenye Kujitafakari na Urejeshaji wa Makosa

Wakala uliopo hapa chini ameagizwa kujaribu mfumo mkuu wa ndege kwanza, kutambua makosa, na kwa uwazi kurejea kwenye mfumo wa akiba. Baada ya kila jibu hutathmini kwa kifupi ikiwa limejibu kikamilifu swali la mtumiaji.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mfano wa Kujitathmini

Sehemu nyingine ya metacognition ni **kujitathmini**: wakala tofauti (au wakala huyo katika mpitio wa pili) hupitia jibu kwa kuangalia ukamilifu, usahihi, na jinsi linavyosaidia.

Hapo chini tunaunda wakala `ResponseEvaluator` ambaye anatoa alama kwa majibu ya wakala wa usafiri kwa vipimo vitatu.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Muhtasari

Katika somo hili ulijifunza jinsi ya kujenga **maajenti wa metakognitifu** kwa kutumia Microsoft Agent Framework:

- **Kujitafakari**: Maajenti ambao hufuatilia mchakato wao wa kufikiri na huwasilisha kwa uwazi kilichotokea.
- **Urejeshaji wa makosa kwa mbadala**: Mfano wa zana kuu + nakala ambapo maajenti hugundua kushindwa (kwa mfano, makosa ya 404) na kwa moja kwa moja hujaribu chanzo mbadala.
- **Kujiangalia**: Maajenti wa tathmini huru wanaopima majibu kwa ukamilifu, usahihi, na uwezo wa kusaidia.

Mifano hii inafanya maajenti kuwa thabiti zaidi, wazi, na wa kuaminika — sifa muhimu kwa kuanzishwa katika mazingira ya uzalishaji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Tamko:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuwa sahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au kutokuwa sahihi. Nyaraka ya asili kwa lugha yake ya mama inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, inashauriwa kutumia tafsiri ya kitaalamu ya binadamu. Sisi hatuwajibiki kwa kutoelewana au tafsiri potofu zitokanazo na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
